In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
import numpy as onp
import jax
import jax.numpy as jnp
from tqdm import tqdm

from msmjax.utils.benchmarking import (
    eval_lammps_pppm,
    path_input_structures,
    calc_nonperiodic_reference_results,
)

In [2]:
LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

# Non-periodic

In [4]:
n_particles = 10000
outdir = Path("nonperiodic/")

outdir.mkdir(parents=True, exist_ok=True)
structures = onp.load(path_input_structures / f"structures_{n_particles}.npz")
n_structures = structures["positions"].shape[0]

all_energies = onp.full(n_structures, onp.nan)
all_forces = onp.full((n_structures, n_particles, 3), onp.nan)
all_chargegrads = onp.full((n_structures, n_particles), onp.nan)
all_stresses = onp.full((n_structures, 6), onp.nan)
for i in tqdm(range(n_structures)):
    pos = structures["positions"][i].astype(onp.float64)
    chg = structures["charges"][i].astype(onp.float64)
    cll = structures["cells"][i].astype(onp.float64)
    energy, forces, chargegrad, stress = calc_nonperiodic_reference_results(
        pos, chg, cll
    )
    all_energies[i] = energy
    all_forces[i] = forces
    all_chargegrads[i] = chargegrad
    all_stresses[i] = stress

onp.savez_compressed(
    outdir / "structures.npz",
    positions=structures["positions"].astype(onp.float64),
    charges=structures["charges"].astype(onp.float64),
    cells=structures["cells"].astype(onp.float64),
)
onp.savez_compressed(
    outdir / "reference_results.npz",
    energies=all_energies,
    forces=all_forces,
    chargegrads=all_chargegrads,
    stresses=all_stresses,
)

100%|██████████| 11/11 [00:04<00:00,  2.74it/s]


# Periodic

In [14]:
n_particles = 2000
outdir = Path("periodic/")

outdir.mkdir(parents=True, exist_ok=True)
structures = onp.load(path_input_structures / f"structures_{n_particles}.npz")
n_structures = structures["positions"].shape[0]

all_energies = onp.full(n_structures, onp.nan)
all_forces = onp.full((n_structures, n_particles, 3), onp.nan)
all_chargegrads = onp.full((n_structures, n_particles), onp.nan)
all_stresses = onp.full((n_structures, 6), onp.nan)
for i in tqdm(range(n_structures)):
    pos = structures["positions"][i].astype(onp.float64)
    chg = structures["charges"][i].astype(onp.float64)
    cll = structures["cells"][i].astype(onp.float64)
    energy, forces, chargegrad, stress = eval_lammps_pppm(
        pos,
        chg,
        cll,
        LAMMPS_EXECUTABLE,
        accuracy=1.0e-8,
        max_neighbors_one_atom=10000,
    )
    all_energies[i] = energy
    all_forces[i] = forces
    all_chargegrads[i] = chargegrad
    all_stresses[i] = stress

onp.savez_compressed(
    outdir / "structures.npz",
    positions=structures["positions"].astype(onp.float64),
    charges=structures["charges"].astype(onp.float64),
    cells=structures["cells"].astype(onp.float64),
)
onp.savez_compressed(
    outdir / "reference_results.npz",
    energies=all_energies,
    forces=all_forces,
    chargegrads=all_chargegrads,
    stresses=all_stresses,
)

100%|██████████| 11/11 [00:04<00:00,  2.53it/s]
